<a href="https://colab.research.google.com/github/alifihsan7/kelompok9-image-captioning/blob/main/notebooks/02_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
from pathlib import Path

# 1. Menetapkan asumsi jalur folder proyek di Google Drive
# Menangani variasi penamaan default MyDrive / My Drive di Colab
CANDIDATE_PATHS = [
    Path("/content/drive/MyDrive/Kelompok9"),
    Path("/content/drive/My Drive/Kelompok9"),
]

project_dir = None
for path in CANDIDATE_PATHS:
    if path.is_dir():
        project_dir = path
        break

# 2 & 3. Pemeriksaan keberadaan folder utama dan subfolder src
if project_dir is None:
    # 5. Pesan jika folder utama tidak ditemukan
    print("❌ ERROR: Folder utama 'Kelompok9' tidak ditemukan di Google Drive.")
    print("Pemeriksaan yang perlu dilakukan:")
    print("1. Pastikan mount Google Drive berhasil (`drive.mount('/content/drive')`).")
    print("2. Jika 'Kelompok9' adalah folder bersama (Shared Folder), pastikan sudah menambahkan pintasan ke Drive ('Add shortcut to Drive') pada akun Anda.")
    print("3. Pastikan penulisan nama folder tepat 'Kelompok9' (sensitif huruf besar/kecil).")
else:
    # 2. Menentukan lokasi subfolder src
    src_dir = project_dir / "src"

    # 4. Menampilkan lokasi yang ditemukan
    print(f"✅ Folder utama proyek ditemukan : {project_dir}")

    # 3 & 5. Pemeriksaan ketersediaan subfolder src
    if src_dir.is_dir():
        print(f"✅ Subfolder src ditemukan        : {src_dir}")
        print("\nStatus: Jalur valid. Anda siap menyimpan 'src/model.py'.")
    else:
        print(f"❌ ERROR: Subfolder 'src' tidak ditemukan di dalam {project_dir}.")
        print("Peringatan: Folder baru tidak dibuat secara otomatis.")
        print(f"Silakan buat folder 'src' di Google Drive secara manual pada lokasi: {project_dir}")

✅ Folder utama proyek ditemukan : /content/drive/MyDrive/Kelompok9
✅ Subfolder src ditemukan        : /content/drive/MyDrive/Kelompok9/src

Status: Jalur valid. Anda siap menyimpan 'src/model.py'.


Mengatur Direktori Kerja dan Menyimpan src/model.py

In [11]:
import os
import sys

# Pindah ke direktori utama proyek di Drive yang sudah terverifikasi sebelumnya
os.chdir(project_dir)

# Tambahkan direktori proyek ke sys.path agar impor modul 'src' berjalan lancar
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print(f"Direktori kerja aktif saat ini: {os.getcwd()}")

Direktori kerja aktif saat ini: /content/drive/MyDrive/Kelompok9


In [12]:
%%writefile src/model.py
"""
src/model.py
Modul arsitektur Decoder LSTM untuk Image Captioning.
Mengikuti spesifikasi Kontrak Teknis Data V1 (Kelompok 9).
"""

from typing import List
import torch
import torch.nn as nn


class CaptionDecoder(nn.Module):
    """
    Decoder LSTM untuk Image Captioning.

    Arsitektur:
        - Proyeksi linear fitur visual [B, feature_dim] ke status awal LSTM (h0, c0).
        - Lapisan embedding token kata dengan penanganan padding_idx.
        - LSTM 1 lapis (batch_first=True) untuk pemrosesan sekuens teks.
        - Regulerisasi Dropout pada representasi status tersembunyi.
        - Lapisan linear proyeksi akhir ke dimensi kosakata (raw logits).

    Args:
        vocab_size (int): Jumlah total kosakata unik (V).
        feature_dim (int, optional): Dimensi vektor fitur visual encoder ResNet-50. Default: 2048.
        embed_dim (int, optional): Dimensi representasi vektor kata. Default: 256.
        hidden_size (int, optional): Dimensi ruang status tersembunyi (hidden & cell state). Default: 512.
        dropout_p (float, optional): Probabilitas dropout regulerisasi. Default: 0.5.
        pad_idx (int, optional): Indeks token khusus <pad>. Default: 0.
    """

    def __init__(
        self,
        vocab_size: int,
        feature_dim: int = 2048,
        embed_dim: int = 256,
        hidden_size: int = 512,
        dropout_p: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.feature_dim = feature_dim
        self.embed_dim = embed_dim
        self.hidden_size = hidden_size
        self.dropout_p = dropout_p
        self.pad_idx = pad_idx

        # 1. Proyeksi fitur visual ke hidden state (h0) dan cell state (c0) awal
        # [B, feature_dim] -> [B, hidden_size]
        self.init_h = nn.Linear(feature_dim, hidden_size)
        self.init_c = nn.Linear(feature_dim, hidden_size)

        # 2. Lapisan penyematan kata (word embedding)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx,
        )

        # 3. Lapisan recurrent LSTM 1 lapis
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        # 4. Lapisan regulerisasi dropout
        self.dropout = nn.Dropout(p=dropout_p)

        # 5. Lapisan linear keluaran menuju ruang kosakata
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(
        self, features: torch.Tensor, captions_in: torch.Tensor
    ) -> torch.Tensor:
        """
        Menjalankan satu lintasan komputasi maju menggunakan skema Teacher Forcing.

        Bentuk Tensor:
            - features:    torch.FloatTensor [B, feature_dim]
            - captions_in: torch.LongTensor  [B, L]
            - logits:      torch.FloatTensor [B, L, vocab_size]

        Args:
            features (torch.Tensor): Representasi visual dari encoder berdimensi [B, feature_dim].
            captions_in (torch.Tensor): Sekuens token masukan berdimensi [B, L].

        Returns:
            torch.Tensor: Logits (skor mentah sebelum aktivasi softmax) berdimensi [B, L, vocab_size].
        """
        # 1. Inisialisasi h0 dan c0 dari fitur visual
        # [B, feature_dim] -> [B, hidden_size] -> [1, B, hidden_size]
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)

        # 2. Pemetaan token teks ke ruang kontinu
        # [B, L] -> [B, L, embed_dim]
        embedded = self.embedding(captions_in)

        # 3. Propagasi sekuensial LSTM
        # lstm_out berdimensi [B, L, hidden_size]
        lstm_out, _ = self.lstm(embedded, (h0, c0))

        # 4. Penerapan dropout pada luaran hidden state
        out = self.dropout(lstm_out)

        # 5. Proyeksi skor mentah ke kosakata tanpa softmax
        # [B, L, hidden_size] -> [B, L, vocab_size]
        logits = self.fc_out(out)

        return logits

    def generate_greedy(
        self,
        feature: torch.Tensor,
        max_len: int = 30,
        start_token_id: int = 1,
        end_token_id: int = 2,
        pad_token_id: int = 0,
    ) -> List[int]:
        """
        Membangkitkan token deskripsi gambar secara greedy langkah demi langkah (autoregresif).

        Perbedaan Batas Panjang:
            - max_len=30 di sini menentukan batas maksimal langkah inferensi (maksimal 30 token prediksi).
            - Ini berbeda dari panjang sekuens training T=30 yang mencakup <start>, kata isi, <end>, dan <pad>.

        Bentuk Tensor Masukan:
            - feature: [feature_dim] (1D) atau [1, feature_dim] (2D)

        Args:
            feature (torch.Tensor): Fitur visual citra tunggal berdimensi [feature_dim] atau [1, feature_dim].
            max_len (int, optional): Batas maksimal langkah prediksi kata. Default: 30.
            start_token_id (int, optional): ID token khusus <start>. Default: 1.
            end_token_id (int, optional): ID token khusus <end>. Default: 2.
            pad_token_id (int, optional): ID token khusus <pad>. Default: 0.

        Returns:
            List[int]: Daftar indeks kata hasil inferensi (tanpa token <start>, <end>, maupun <pad>).

        Raises:
            ValueError: Jika bentuk tensor fitur tidak sesuai dengan dimensi [feature_dim] atau [1, feature_dim].
        """
        # 1. Validasi bentuk tensor input secara ketat
        if feature.dim() == 1:
            if feature.shape[0] != self.feature_dim:
                raise ValueError(
                    f"Fitur gambar 1D harus berukuran [{self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
            feature = feature.unsqueeze(0)  # Menjadi [1, feature_dim]
        elif feature.dim() == 2:
            if feature.shape != (1, self.feature_dim):
                raise ValueError(
                    f"Fitur gambar 2D harus berukuran [1, {self.feature_dim}], "
                    f"tetapi diterima bentuk {list(feature.shape)}."
                )
        else:
            raise ValueError(
                f"Fitur gambar harus berdimensi 1D [{self.feature_dim}] atau 2D [1, {self.feature_dim}], "
                f"tetapi diterima tensor {feature.dim()}D dengan bentuk {list(feature.shape)}."
            )

        # 2. Penyelarasan device antara model, fitur, dan tensor token
        target_device = next(self.parameters()).device
        if feature.device != target_device:
            feature = feature.to(target_device)

        # 3. Manajemen status mode pelatihan (eval & restore via try/finally)
        was_training = self.training
        self.eval()

        generated_tokens: List[int] = []

        try:
            with torch.no_grad():
                # Inisialisasi status tersembunyi awal dari fitur gambar
                # [1, feature_dim] -> [1, hidden_size] -> [1, 1, hidden_size]
                h = self.init_h(feature).unsqueeze(0)
                c = self.init_c(feature).unsqueeze(0)

                # Token pemantik awal: <start> berdimensi [1, 1]
                current_token = torch.tensor(
                    [[start_token_id]], dtype=torch.long, device=target_device
                )

                # Iterasi prediksi autoregresif hingga max_len langkah
                for _ in range(max_len):
                    # Embedding token saat ini: [1, 1] -> [1, 1, embed_dim]
                    emb = self.embedding(current_token)

                    # Propagasi recurrent 1 langkah waktu, perbarui h dan c
                    lstm_out, (h, c) = self.lstm(emb, (h, c))

                    # Proyeksi ke skor mentah kosakata: [1, vocab_size]
                    logits = self.fc_out(lstm_out.squeeze(1))

                    # Masking token PAD dan START agar tidak pernah terpilih
                    logits[0, pad_token_id] = -float("inf")
                    logits[0, start_token_id] = -float("inf")

                    # Pemilihan kata greedy (skor tertinggi)
                    pred_token = torch.argmax(logits, dim=-1).item()

                    # Hentikan jika mencapai penutup <end> tanpa memasukkannya ke hasil
                    if pred_token == end_token_id:
                        break

                    generated_tokens.append(pred_token)

                    # Siapkan input untuk langkah waktu berikutnya
                    current_token = torch.tensor(
                        [[pred_token]], dtype=torch.long, device=target_device
                    )
        finally:
            # Pulihkan status pelatihan awal
            self.train(was_training)

        return generated_tokens


Overwriting src/model.py


In [13]:
import importlib
import torch
import torch.nn as nn

# Reload modul agar perubahan generate_greedy termuat
import src.model
importlib.reload(src.model)
from src.model import CaptionDecoder

print("--- MEMULAI PENGUJIAN UNIT GENERATE_GREEDY ---")

# 1. Inisialisasi Model Uji
torch.manual_seed(42)
vocab_size = 3000
feature_dim = 2048
model = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim)

# Menentukan device aktif model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------------------------------------------------------------
# Uji 1: Validasi Bentuk Input 1D dan 2D serta Penolakan Bentuk Salah
# -------------------------------------------------------------------------
f_1d = torch.randn(feature_dim, device=device)
f_2d = torch.randn(1, feature_dim, device=device)

tokens_1d = model.generate_greedy(f_1d, max_len=10)
tokens_2d = model.generate_greedy(f_2d, max_len=10)

assert isinstance(tokens_1d, list), "Keluaran input 1D harus berupa list"
assert isinstance(tokens_2d, list), "Keluaran input 2D harus berupa list"
print("✅ Uji 1a: Input 1D [2048] dan 2D [1, 2048] berhasil diproses.")

# Uji penolakan bentuk input invalid
shape_error_caught = 0
for bad_shape in [(1024,), (32, 2048), (1, 1, 2048)]:
    try:
        bad_tensor = torch.randn(*bad_shape, device=device)
        model.generate_greedy(bad_tensor)
    except ValueError as e:
        shape_error_caught += 1

assert shape_error_caught == 3, "Model harus menolak bentuk selain [2048] dan [1, 2048]"
print("✅ Uji 1b: Penanganan ValueError untuk input tidak valid berfungsi dengan benar.")

# -------------------------------------------------------------------------
# Uji 2: Pengecekan Masking PAD (0), START (1), dan Pengecualian END (2)
# -------------------------------------------------------------------------
# Hasil token keluaran tidak boleh mengandung token 0, 1, maupun 2
for token_list in [tokens_1d, tokens_2d]:
    assert 0 not in token_list, "Error: Token <pad> (0) ditemukan pada hasil!"
    assert 1 not in token_list, "Error: Token <start> (1) ditemukan pada hasil!"
    assert 2 not in token_list, "Error: Token <end> (2) ditemukan pada hasil!"
    assert all(isinstance(t, int) for t in token_list), "Seluruh elemen harus bertipe int."
print("✅ Uji 2: Output list[int] bersih tanpa token PAD (0), START (1), maupun END (2).")

# -------------------------------------------------------------------------
# Uji 3: Batas Panjang & Validasi Masking PAD/START (Kasus Terkontrol)
# -------------------------------------------------------------------------
# 1. Inisialisasi model uji terpisah
model_controlled_3 = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)

with torch.no_grad():
    # 2. Nolkan bobot fc_out agar skor murni ditentukan oleh bias (independen dari representasi LSTM)
    model_controlled_3.fc_out.weight.zero_()

    # 3. Konfigurasi bias terkontrol
    # Set seluruh token (termasuk END=2 dan token lainnya) ke nilai rendah
    model_controlled_3.fc_out.bias.fill_(-10.0)

    # PAD (0) dan START (1) diberi bias tertinggi
    model_controlled_3.fc_out.bias[0] = 100.0  # PAD (<pad>=0)
    model_controlled_3.fc_out.bias[1] = 90.0   # START (<start>=1)

    # Token isi target (ID 4) menjadi kandidat tertinggi berikutnya
    model_controlled_3.fc_out.bias[4] = 50.0

# 4. Jalankan inferensi dengan batas langkah max_len=15
max_test_len = 15
tokens_uji3 = model_controlled_3.generate_greedy(f_2d, max_len=max_test_len)

# 5. Validasi:
# - Jika masking gagal, argmax akan memilih token 0 (<pad>) atau token 1 (<start>)
# - Karena di-mask ke -inf, argmax memilih token 4 pada setiap langkah
# - Karena END (2) tidak terpilih, inferensi berhenti tepat pada max_len langkah
expected_tokens = [4] * max_test_len

assert tokens_uji3 == expected_tokens, (
    f"Keluaran tidak sesuai! Diharapkan {expected_tokens}, tetapi diperoleh {tokens_uji3}"
)
assert len(tokens_uji3) == max_test_len, (
    f"Panjang keluaran salah! Diharapkan {max_test_len}, didapat {len(tokens_uji3)}"
)

print(f"✅ Uji 3: Kasus terkontrol lolos — masking PAD/START terbukti aktif dan loop berhenti tepat pada max_len={max_test_len} ({tokens_uji3[:3]}...).")

# -------------------------------------------------------------------------
# Uji 4: Berhenti Ketika Token END Terpilih (Kasus Terkontrol)
# -------------------------------------------------------------------------
# Membuat model kontrol di mana token <end> (indeks 2) dipaksa memiliki bobot/bias tertinggi
model_controlled = CaptionDecoder(vocab_size=vocab_size, feature_dim=feature_dim).to(device)

with torch.no_grad():
    # Set bias output sangat negatif, kecuali untuk token <end>
    model_controlled.fc_out.bias.fill_(-1000.0)
    model_controlled.fc_out.bias[2] = 1000.0  # Token END dipastikan terpilih pada langkah pertama

tokens_end_immediate = model_controlled.generate_greedy(f_2d, max_len=30)
assert len(tokens_end_immediate) == 0, f"Seharusnya langsung berhenti tanpa menambah token, didapat len={len(tokens_end_immediate)}"
assert 2 not in tokens_end_immediate, "Token END tidak boleh dimasukkan ke dalam list"
print("✅ Uji 4: Kasus terkontrol berhasil berhenti seketika saat token <end> terpilih.")

# -------------------------------------------------------------------------
# Uji 5: Pemulihan Status Mode Pelatihan (train vs eval)
# -------------------------------------------------------------------------
# Kasus 5a: Mode awal train -> harus kembali train
model.train()
assert model.training is True
_ = model.generate_greedy(f_2d, max_len=5)
assert model.training is True, "Gagal memulihkan model ke mode train!"

# Kasus 5b: Mode awal eval -> harus tetap eval
model.eval()
assert model.training is False
_ = model.generate_greedy(f_2d, max_len=5)
assert model.training is False, "Gagal mempertahankan model dalam mode eval!"
print("✅ Uji 5: Pemulihan status model (train maupun eval) via try/finally terverifikasi aman.")

# -------------------------------------------------------------------------
# Uji 6: Integritas Gradien (Tidak Membuat Maupun Mengubah Gradien)
# -------------------------------------------------------------------------
# Siapkan gradien pada model terlebih dahulu (mensimulasikan backward pass sebelumnya)
model.train()
dummy_feat_train = torch.randn(2, feature_dim, device=device)
dummy_captions_train = torch.randint(0, vocab_size, (2, 29), device=device)
dummy_logits = model(dummy_feat_train, dummy_captions_train)
dummy_loss = dummy_logits.sum()
dummy_loss.backward()

# Ambil salinan persis nilai gradien yang ada
grad_h_before = model.init_h.weight.grad.clone()
assert grad_h_before is not None, "Gradien awal model harus ada."

# Jalankan inferensi greedy menggunakan tensor masukan yang memiliki requires_grad=True
f_grad_test = torch.randn(feature_dim, device=device, requires_grad=True)
_ = model.generate_greedy(f_grad_test, max_len=5)

# Periksa bahwa tensor input tidak menerima gradien baru
assert f_grad_test.grad is None, "Input tensor inferensi tidak boleh menghasilkan gradien!"

# Periksa bahwa gradien bobot model yang sudah ada TIDAK berubah sedikit pun
assert torch.equal(model.init_h.weight.grad, grad_h_before), "Gradien model yang sudah ada terubah selama inferensi!"
print("✅ Uji 6: Inferensi tidak membuat gradien baru dan tidak mengubah gradien yang sudah ada.")

print("\n--- SEMUA PENGUJIAN UNIT GENERATE_GREEDY BERHASIL DILALUI ---")


--- MEMULAI PENGUJIAN UNIT GENERATE_GREEDY ---
✅ Uji 1a: Input 1D [2048] dan 2D [1, 2048] berhasil diproses.
✅ Uji 1b: Penanganan ValueError untuk input tidak valid berfungsi dengan benar.
✅ Uji 2: Output list[int] bersih tanpa token PAD (0), START (1), maupun END (2).
✅ Uji 3: Kasus terkontrol lolos — masking PAD/START terbukti aktif dan loop berhenti tepat pada max_len=15 ([4, 4, 4]...).
✅ Uji 4: Kasus terkontrol berhasil berhenti seketika saat token <end> terpilih.
✅ Uji 5: Pemulihan status model (train maupun eval) via try/finally terverifikasi aman.
✅ Uji 6: Inferensi tidak membuat gradien baru dan tidak mengubah gradien yang sudah ada.

--- SEMUA PENGUJIAN UNIT GENERATE_GREEDY BERHASIL DILALUI ---


Verifikasi Fisik File dan Uji Impor Modul

In [14]:
from pathlib import Path
import importlib

target_file = project_dir / "src" / "model.py"

if target_file.is_file():
    print(f"✅ Berkas berhasil diverifikasi di: {target_file}")

    # Impor kelas CaptionDecoder dari berkas src/model.py
    import src.model
    importlib.reload(src.model)  # Memastikan versi terbaru yang dimuat jika ada perubahan
    from src.model import CaptionDecoder

    print("✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.")
else:
    print(f"❌ ERROR: Berkas {target_file} belum ditemukan di Google Drive.")

✅ Berkas berhasil diverifikasi di: /content/drive/MyDrive/Kelompok9/src/model.py
✅ Berhasil mengimpor 'CaptionDecoder' dari 'src.model'.


Uji Fungsional Tensor Dummy, Loss, dan Aliran Gradien

In [15]:
import torch
import torch.nn as nn

# 1. Parameter tensor pengujian
B = 32
feature_dim = 2048
L = 29
vocab_size = 3000

# 2. Penyiapan tensor dummy
torch.manual_seed(42)
dummy_features = torch.randn(B, feature_dim, dtype=torch.float32)
dummy_captions_in = torch.randint(0, vocab_size, (B, L), dtype=torch.long)
dummy_targets = torch.randint(0, vocab_size, (B, L), dtype=torch.long)

# 3. Inisialisasi model
model = CaptionDecoder(
    vocab_size=vocab_size,
    feature_dim=feature_dim,
    embed_dim=256,
    hidden_size=512,
    dropout_p=0.5,
    pad_idx=0
)

# 4. Uji forward pass
logits = model(dummy_features, dummy_captions_in)
print(f"Bentuk tensor logits   : {list(logits.shape)}")
assert logits.shape == (B, L, vocab_size), f"Dimensi logits tidak cocok! Diharapkan {(B, L, vocab_size)}, didapat {logits.shape}"
print("✅ Bentuk logits sesuai kontrak [B, L, vocab_size].")

# 5. Uji kalkulasi loss (perataan sesuai kontrak: [B*L, V] dan [B*L])
criterion = nn.CrossEntropyLoss(ignore_index=0)
loss = criterion(logits.reshape(-1, vocab_size), dummy_targets.reshape(-1))
print(f"Nilai loss dummy       : {loss.item():.4f}")

# 6. Uji backward pass dan verifikasi gradien
model.zero_grad()
loss.backward()

grad_h = model.init_h.weight.grad
grad_c = model.init_c.weight.grad
grad_fc = model.fc_out.weight.grad

assert grad_h is not None and grad_h.norm().item() > 0, "Gradien pada init_h tidak mengalir!"
assert grad_c is not None and grad_c.norm().item() > 0, "Gradien pada init_c tidak mengalir!"
assert grad_fc is not None and grad_fc.norm().item() > 0, "Gradien pada fc_out tidak mengalir!"

print(f"Norm gradien init_h    : {grad_h.norm().item():.6f} (mengalir)")
print(f"Norm gradien init_c    : {grad_c.norm().item():.6f} (mengalir)")
print(f"Norm gradien fc_out    : {grad_fc.norm().item():.6f} (mengalir)")
print("\n✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.")

Bentuk tensor logits   : [32, 29, 3000]
✅ Bentuk logits sesuai kontrak [B, L, vocab_size].
Nilai loss dummy       : 8.0069
Norm gradien init_h    : 0.034732 (mengalir)
Norm gradien init_c    : 0.064243 (mengalir)
Norm gradien fc_out    : 0.116075 (mengalir)

✅ Seluruh pengujian forward pass, perataan loss, dan aliran gradien BERHASIL.
